# Starter notebook: Real Estate Price Prediction

1. Load the data
2. Split `train.csv` into your own train and test parts
3. Train a model with all features
4. Keep only the significant features and retrain
5. Measure RMSE on your test part
6. Retrain on all of `train.csv` and create `submission.csv`

**Your goal:** beat this notebook's leaderboard score.

## Step 1. Load the data

In [ ]:
import glob, os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Find the competition folder on Kaggle
folder = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])

train = pd.read_csv(f'{folder}/train.csv')
test = pd.read_csv(f'{folder}/test.csv')

TARGET = 'price_per_area'
ALL_FEATURES = [c for c in train.columns if c not in ('id', TARGET)]

train.head()

## Step 2. Split train.csv into your own train and test parts

`test.csv` has no prices, so we hold back 20% of `train.csv` to check our model ourselves.

In [ ]:
my_train, my_test = train_test_split(train, test_size=0.2, random_state=42)

print('My train:', len(my_train), '  My test:', len(my_test))

## Step 3. Train a model with all features

What to look at in the summary:
- **coef**: how much the price changes when the feature goes up by 1
- **t**: coef ÷ std err. Far from 0 (beyond ±2) = strong evidence
- **P>|t|** (p-value): below 0.05 = significant (keep); above 0.05 = weak (drop)
- **R-squared**: how much of the variation the model explains (0 to 1)
- **Adj. R-squared**: R-squared with a penalty for extra features

In [ ]:
model_all = sm.OLS(my_train[TARGET], sm.add_constant(my_train[ALL_FEATURES])).fit()
print(model_all.summary())

## Step 4. Keep only the significant features and retrain

In [ ]:
pvalues = model_all.pvalues.drop('const')
FEATURES = list(pvalues[pvalues < 0.05].index)

print('Keep:', FEATURES)
print('Drop:', list(pvalues[pvalues >= 0.05].index))

model = sm.OLS(my_train[TARGET], sm.add_constant(my_train[FEATURES])).fit()
print(model.summary())

print(f'Adj. R-squared, all features: {model_all.rsquared_adj:.3f}')
print(f'Adj. R-squared, kept features: {model.rsquared_adj:.3f}')

## Step 5. Measure RMSE on your test part

In [ ]:
pred = model.predict(sm.add_constant(my_test[FEATURES]))
rmse = np.sqrt(mean_squared_error(my_test[TARGET], pred))

baseline_pred = [my_train[TARGET].mean()] * len(my_test)
baseline_rmse = np.sqrt(mean_squared_error(my_test[TARGET], baseline_pred))

print(f'My RMSE       : {rmse:.2f}')
print(f'Baseline RMSE : {baseline_rmse:.2f}   (always guessing the average)')

## Step 6. Retrain on all of train.csv and create submission.csv

Then click **Submit** (or download `submission.csv` and upload it on the competition page).

In [ ]:
final_model = sm.OLS(train[TARGET], sm.add_constant(train[FEATURES])).fit()

submission = pd.DataFrame({
    'id': test['id'],
    TARGET: final_model.predict(sm.add_constant(test[FEATURES])),
})
submission.to_csv('submission.csv', index=False)
submission.head()